In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib


In [48]:
print("=== BASELINE MODEL : 10-MATCH ROLLING AVERAGE ===")

print("Total rows before baseline:", len(df))

df = df.sort_values(["batter", "date"])

df["baseline_pred"] = (
    df.groupby("batter")["runs"]
      .shift(1)
      .rolling(window=10, min_periods=1)
      .mean()
)

baseline_df = df.dropna(subset=["baseline_pred", "target_next_runs"])

print("Rows used for baseline evaluation:", len(baseline_df))

rmse = np.sqrt(mean_squared_error(
    baseline_df["target_next_runs"],
    baseline_df["baseline_pred"]
))
mae = mean_absolute_error(
    baseline_df["target_next_runs"],
    baseline_df["baseline_pred"]
)
r2 = r2_score(
    baseline_df["target_next_runs"],
    baseline_df["baseline_pred"]
)

print(f"Baseline RMSE : {rmse:.2f}")
print(f"Baseline MAE  : {mae:.2f}")
print(f"Baseline R2   : {r2:.3f}")


=== BASELINE MODEL : 10-MATCH ROLLING AVERAGE ===
Total rows before baseline: 54
Rows used for baseline evaluation: 37
Baseline RMSE : 28.04
Baseline MAE  : 18.96
Baseline R2   : -0.221


In [49]:
print("\n=== TRAIN TEST SPLIT ===")

df = df.dropna(subset=["target_next_runs"])

X = df[features]
y = df["target_next_runs"]

print("Features used:", features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)



=== TRAIN TEST SPLIT ===
Features used: ['runs', 'balls', 'fours', 'sixes', 'form_last_5', 'venue_avg', 'opponent_avg_run', 'career_avg_runs', 'career_strike_rate']
X_train shape: (43, 9)
X_test shape : (11, 9)
y_train shape: (43,)
y_test shape : (11,)


In [50]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{model_name} Performance")
    print(f"RMSE : {rmse:.2f}")
    print(f"MAE  : {mae:.2f}")
    print(f"R2   : {r2:.3f}")


In [51]:
print("\n=== RANDOM FOREST MODEL ===")

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

evaluate(y_test, rf_pred, "Random Forest")



=== RANDOM FOREST MODEL ===

Random Forest Performance
RMSE : 14.85
MAE  : 10.84
R2   : 0.044


In [52]:
print("\n=== XGBOOST MODEL ===")

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

evaluate(y_test, xgb_pred, "XGBoost")



=== XGBOOST MODEL ===

XGBoost Performance
RMSE : 16.22
MAE  : 14.00
R2   : -0.141


In [53]:
print("\n=== LIGHTGBM MODEL ===")

lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

lgbm.fit(X_train, y_train)
lgbm_pred = lgbm.predict(X_test)

evaluate(y_test, lgbm_pred, "LightGBM")



=== LIGHTGBM MODEL ===
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000040 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 86
[LightGBM] [Info] Number of data points in the train set: 43, number of used features: 7
[LightGBM] [Info] Start training from score 26.883721
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

In [54]:
print("\n=== XGBOOST HYPERPARAMETER TUNING ===")

param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05],
    "n_estimators": [300, 500]
}

grid = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best RMSE:", -grid.best_score_)
print("Best Parameters:", grid.best_params_)

best_xgb = grid.best_estimator_



=== XGBOOST HYPERPARAMETER TUNING ===
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best RMSE: 32.700308486270416
Best Parameters: {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 300}


In [55]:
print("\n=== SHAP FEATURE IMPORTANCE ===")

explainer = shap.Explainer(best_xgb, X_train)
shap_values = explainer(X_train)

shap_df = pd.DataFrame({
    "feature": X_train.columns,
    "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)
}).sort_values(by="mean_abs_shap", ascending=False)

print("Top 5 Important Features:")
print(shap_df.head(5))



=== SHAP FEATURE IMPORTANCE ===
Top 5 Important Features:
              feature  mean_abs_shap
5           venue_avg      12.522650
1               balls       8.949816
8  career_strike_rate       6.033636
0                runs       4.776592
2               fours       1.480984


In [57]:
joblib.dump(best_xgb, "xgb_model.joblib")
print("Model saved successfully as xgb_model.joblib")


Model saved successfully as xgb_model.joblib
